In [1]:
import h5py
import numpy as np
import networkx as nx
import pandas as pd
#from pygsp import graphs, filters, plotting
import geopandas as gpd
import itertools
from shapely.geometry import Point
import torch
import toponetx as tnx

from sklearn.neighbors import radius_neighbors_graph,kneighbors_graph

In [2]:
from topomodelx.base.conv import Conv

In [2]:
#eval_mask = pd.read_hdf("full437.h5", mode="r", key="pm25")
pm25 = pd.read_hdf("small36.h5", mode="r", key="pm25")
stations = pd.read_hdf("small36.h5", mode="r", key="stations")

stations = gpd.GeoDataFrame(stations)
stations[["latitude", "longitude"]].head()

/Users/zhaobo/anaconda3/envs/tmx/lib/python3.11/site-packages/pandas/core/frame.py:717: DeprecationWarning: Passing a BlockManager to GeoDataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(


,latitude,longitude
sensor_id,,
1001,40.090679,116.173553
1002,40.003948,116.205307
1003,39.914410,116.184242
1004,39.815128,116.171150
1005,39.742767,116.136047


In [3]:
from sklearn.metrics.pairwise import haversine_distances
from sklearn.metrics import DistanceMetric
from math import radians

def haversine_distances_km(x, y):
    x_in_radians = [radians(_) for _ in x]
    y_in_radians = [radians(_) for _ in y]
    result = haversine_distances([x_in_radians, y_in_radians])
    
    return result[0,1] * 6371000/1000

metric = DistanceMetric.get_metric("pyfunc", func = haversine_distances_km)

#超过radius距离就不算了
A = radius_neighbors_graph(stations[["latitude", "longitude"]], radius = 40,mode='distance',include_self=True, n_jobs=-1, metric="pyfunc",metric_params={"func":haversine_distances_km})

distance_matrix = A.toarray()
distance_matrix[distance_matrix != 0] = 1

/Users/zhaobo/anaconda3/envs/tmx/lib/python3.11/site-packages/pandas/core/frame.py:717: DeprecationWarning: Passing a BlockManager to GeoDataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
/Users/zhaobo/anaconda3/envs/tmx/lib/python3.11/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(


In [4]:
# convert adjacency matrix to edge set(index)
def to_edge_index(adj):
    edge_index = adj.nonzero().t().contiguous()
    return edge_index

def find_cliques_size_k(G, k):
    all_cliques = set()
    for clique in nx.find_cliques(G):
        if len(clique) == k:
            all_cliques.add(tuple(sorted(clique)))
        elif len(clique) > k:
            for mini_clique in itertools.combinations(clique, k):
                all_cliques.add(tuple(sorted(mini_clique)))
    return list(all_cliques)
    
#adj_t = torch.tensor([[0,1,0,0],[1,0,0,0], [1,0,0,1], [0,0,1,0]])
edge_index = to_edge_index(torch.tensor(distance_matrix))

In [12]:
G = nx.from_numpy_array(distance_matrix)
edge_set = np.array(edge_index).T
face_set = find_cliques_size_k(G,3)
sc = tnx.SimplicialComplex(list(edge_set) + face_set)

#sc.simplices

NameError: name 'nn' is not defined

In [17]:
from torch_geometric.nn import GCNConv
from torch import nn

In [ ]:
nn.Module

In [18]:
class GCGRUcell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.forget_gate = GCNConv(input_size + hidden_size, hidden_size)
        self.update_gate = GCNConv(input_size + hidden_size, hidden_size)
        self.c_gate = GCNConv(input_size + hidden_size, hidden_size)
        
    def forward(self, x, h, edge_index):
        # x --> [batch_size/node_nums, features]
        x_gates = torch.cat([x, h], dim=1)
        r = torch.sigmoid(self.forget_gate(x_gates, edge_index))
        u = torch.sigmoid(self.update_gate(x_gates, edge_index))
        x_c = torch.cat([x, r * h], dim=1)
        c = self.c_gate(x_c, edge_index)
        c = torch.tanh(c)
        return u * h + (1. - u) * c

In [ ]:
class GCRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, n_layers):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        self.gcgru_cells = nn.ModuleList()

        for i in range(self.n_layers):
            self.rnn_cells.append(GCGRUCell(input_size=self.input_size if i == 0 else self.hidden_size,
                                            hidden_size = self.hidden_size))
            
        self.output_layer = nn.Conv2d(self.hidden_size, self.output_size, kernel_size=1)

    def init_hidden_states(self, x):
        return [torch.zeros(size=(x.shape[0], self.hidden_size)).to(x.device) for _ in range(self.n_layers)]

    def single_pass(self, x, h, edge_index):
        pass
        
    def forward(self):
        pass